# 0. Import Required Libraries

In [2]:
import torch
import torchvision
from torchvision.transforms import ToPILImage
from typing import List, Tuple, Optional, Dict, Any

# Import utility functions
import sys
sys.path.append('.')
from my_utils import (
    get_device,
    get_cifar10_data_augmentation,
    create_train_val_split,
    train_model,
    predict as predict_util,
    plot_loss_curves,
    count_parameters,
    create_learning_rate_scheduler
)

show = ToPILImage()

# 1. Prepare Training and Test Data

First, let's introduce the data. The `torchvision` library provides direct download of the CIFAR-10 dataset, so there's no need to manually download it from the web. The `transform` in the code converts raw images into a format suitable for model processing. `trainloader` and `testloader` are used to load the training and test sets, respectively.

In [ ]:
# Set image normalization transforms and download the dataset
import os
os.makedirs('./dataset', exist_ok=True)

# Use utility functions to get data augmentation transforms
transform_train, transform_test = get_cifar10_data_augmentation()

batch_size = 64  # Increased from 4 to 64 for RTX 4090

# Load full training set
full_trainset = torchvision.datasets.CIFAR10(root='./dataset', train=True,
                                        download=True, transform=transform_train)

# Split training set into training and validation sets (90% train, 10% validation)
trainset, valset = create_train_val_split(full_trainset, val_ratio=0.1, seed=42)

# Create data loaders
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                        shuffle=False, num_workers=2)

# Test set
testset = torchvision.datasets.CIFAR10(root='./dataset', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

print(f"Training set size: {len(trainset)}")
print(f"Validation set size: {len(valset)}")
print(f"Test set size: {len(testset)}")

Running the code below, we can see that the CIFAR-10 training set contains 50,000 images, each of size 32×32; each element in the dataset consists of the image data and its corresponding label.

In [ ]:
# Inspect the dataset contents
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(len(trainset))  # Training set size
print(trainset[0][0].size())  # Image size of the first sample
print(trainset[0][1])  # Label of the first sample
print(classes[trainset[0][1]])  # Text label of the first sample

The code below randomly selects an image from the training set for visualization. Note that before displaying, we need to apply the inverse of the `transform` mentioned above:

In [ ]:
# Lines 1-4 have the same meaning as the previous code block
(data, label) = trainset[12]  # Select a sample from the training set for display; can change the number
print(data.size()) 
print(label)  # Label is an integer
print(classes[label])
show((data + 1) / 2).resize((100, 100))  # Reverse the normalization for display

# 2. Define the Network Architecture for Classification

This section defines the network architecture for image classification, implementing the early convolutional neural network LeNet. It consists of two convolutional layers and three fully connected layers. PyTorch provides a convenient interface for defining neural networks, but we won't focus on the specific syntax here — instead, let's observe how data "flows" through the model:

- In the `__init__` method, we initialize the convolutional and fully connected layers as `conv1`, `conv2` and `fc1`, `fc2`, `fc3`.
- Taking `conv1` as an example: `Conv2d(3, 6, 5)` means 3 input channels (RGB), 6 output channels, and a 5×5 kernel.
- Taking `fc1` as an example: `Linear(16 * 5 * 5, 120)` maps from 400 dimensions to 120 dimensions.
- The `forward` method defines the data computation flow through the model. The shape transformations during propagation are annotated in the `forward` method. Finally, we obtain a tensor of shape `[batch_size, 10]`.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    """LeNet-style CNN for CIFAR-10 image classification.
    
    Original architecture adapted for 32×32 RGB images.
    Consists of two convolutional layers and three fully connected layers.
    """
    
    def __init__(self) -> None:
        # Subclasses of nn.Module must call the parent class constructor
        super(Net, self).__init__()

        # Convolutional layer: '3' input channels (RGB), '6' output channels, '5' kernel size
        self.conv1 = nn.Conv2d(3, 6, 5) 
        # Convolutional layer
        self.conv2 = nn.Conv2d(6, 16, 5) 
        # Fully connected layers: y = Wx + b
        self.fc1   = nn.Linear(16*5*5, 120) 
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, 3, 32, 32).
            
        Returns:
            Output tensor of shape (batch_size, 10).
        """
        # Convolution -> ReLU -> MaxPool (ReLU does not change shape)
        # [batch_size, 3, 32, 32] -- conv1 --> [batch_size, 6, 28, 28] -- maxpool --> [batch_size, 6, 14, 14]
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # [batch_size, 6, 14, 14] -- conv2 --> [batch_size, 16, 10, 10] -- maxpool --> [batch_size, 16, 5, 5]
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        # Flatten the 16 * 5 * 5 feature map into [batch_size, 16 * 5 * 5] for the fully connected layers
        x = x.view(x.size()[0], -1) 
        # [batch_size, 16 * 5 * 5] -- fc1 --> [batch_size, 120]
        x = F.relu(self.fc1(x))
        # [batch_size, 120] -- fc2 --> [batch_size, 84]
        x = F.relu(self.fc2(x))
        # [batch_size, 84] -- fc3 --> [batch_size, 10]
        x = self.fc3(x)        
        return x

net = Net()
print(net)

# 3. Model Training and Evaluation Process

With the data prepared and the model defined, we can begin the training process. To optimize a randomly initialized model into a "good" model, we also need to define:

- **Loss function** $\mathcal{L}$: The loss function takes the model's prediction $\hat{y}$ and the true label $y$ as input and outputs a scalar. The smaller this scalar, the better the model fits the data. Our goal is to minimize this loss function $\mathcal{L}(\hat{y}, y)$. Cross-entropy is commonly used as the loss function for classification problems.
- **Optimization method**: To minimize the loss function, we use mathematical optimization methods to find the optimal set of parameters (here, parameters refer to the weights of convolutional layers, fully connected layers, etc., not hyperparameters like batch size). Deep learning typically uses iterative methods, with SGD (Stochastic Gradient Descent) and Adam being common choices.

PyTorch provides various built-in optimizers, so we don't need to implement gradient descent manually.

In [ ]:
from torch import optim

# Loss function: supports label smoothing
label_smoothing = 0.0  # Default: no label smoothing; will be adjusted in Task3
criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

# Optimizer: SGD with momentum (baseline)
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9, weight_decay=0.0)  # No L2 regularization

# Number of training epochs
num_epochs = 20  # Increased from 5 to 20, with early stopping

print(f"Loss function: CrossEntropyLoss (label_smoothing={label_smoothing})")
print(f"Optimizer: SGD (lr=0.001, momentum=0.9, weight_decay=0.0)")
print(f"Epochs: {num_epochs} (with early stopping)")

Below we define the code for the training process. The outer loop controls the number of passes over the entire dataset (i.e., epochs); the inner loop follows these steps:

1. Retrieve data (one batch at a time);
2. Feed data into the network and compute the loss;
3. Compute gradients using the loss function and update parameters via backpropagation.

In [ ]:
def train(
    trainloader: torch.utils.data.DataLoader,
    net: nn.Module,
    num_epochs: int,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    save_path: str,
    valloader: Optional[torch.utils.data.DataLoader] = None,
    early_stopping_patience: Optional[int] = None,
    scheduler_config: Optional[Dict[str, Any]] = None
) -> Tuple[List[float], Optional[List[float]]]:
    """Wrapper around my_utils.train_model for training a neural network.
    
    Args:
        trainloader: DataLoader for training data.
        net: Neural network model.
        num_epochs: Number of training epochs.
        criterion: Loss function.
        optimizer: Optimizer.
        save_path: Directory to save model checkpoints.
        valloader: DataLoader for validation data (optional).
        early_stopping_patience: Patience for early stopping (optional).
        scheduler_config: Learning rate scheduler config from
            create_learning_rate_scheduler (optional).
    
    Returns:
        train_losses: List of average training loss per epoch.
        val_accuracies: List of validation accuracy per epoch (if valloader provided).
    """
    print("Note: Using my_utils.train_model for training")
    return train_model(
        model=net,
        train_loader=trainloader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=num_epochs,
        save_path=save_path,
        val_loader=valloader,
        early_stopping_patience=early_stopping_patience,
        gradient_clip=1.0,
        print_every=1000,
        scheduler_config=scheduler_config
    )

In [ ]:
# 设置设备（GPU如果可用）
device = get_device()
print(f"Using device: {device}")

# 将模型移动到设备
net = Net()
net = net.to(device)

# 创建学习率调度器配置（线性Warm-up + 余弦退火重启）
scheduler_config = create_learning_rate_scheduler(
    optimizer, scheduler_type='cosine',
    total_epochs=num_epochs, initial_lr=0.001
)

# 使用定义的网络进行训练
save_path = 'checkpoints/lenet_baseline'
train_losses, val_accuracies = train(
    trainloader, net, num_epochs, criterion, optimizer, save_path,
    valloader=valloader, early_stopping_patience=10,
    scheduler_config=scheduler_config
)

# 使用工具函数绘制损失曲线
plot_loss_curves(train_losses, val_accuracies, 'loss_curve.png')

print(f"Final training loss: {train_losses[-1]:.4f}")
if val_accuracies:
    print(f"Final validation accuracy: {val_accuracies[-1]:.2f}%")

After training, we have a model that fits the training set well. Next, we evaluate its performance on the test set. The prediction code is similar to the forward pass during training, but does not require computing the loss function (the loss is only needed for updating parameters during training; parameters are fixed during prediction).

The prediction process is as follows:

1. Retrieve data;
2. Forward pass to obtain the model's output;
3. Determine the model's prediction from the output;
4. Compare with the true labels and compute performance metrics.

Note: The model's output, as explained in Section 2, is a tensor of shape `[batch_size, 10]`, where each row represents the relative probability magnitudes for the 10 classes (this output is also called `logits`). To obtain the model's prediction, we take the maximum value along each row — the **position** of the maximum value gives the model's predicted class.

In [ ]:
def predict(testloader: torch.utils.data.DataLoader, net: nn.Module) -> float:
    """Predict and calculate test accuracy (wrapper around my_utils.predict).
    
    Args:
        testloader: DataLoader for test data.
        net: Neural network model.
    
    Returns:
        accuracy: Test accuracy percentage (0-100).
    """
    print("Note: Using my_utils.predict for evaluation")
    return predict_util(net, testloader)

In [20]:
predict(testloader, net)

测试集中的准确率为: 59 %


# Task1: Plot Training Loss Curve

The loss function quantifies how well the model fits the data, helping us understand the training progress. In the `3. Model Training and Evaluation Process` section, add code to record the loss changes during training, save them using an appropriate Python data type, and visualize them using `matplotlib`. You can refer to the code below for plotting. You may overwrite the following code block with your loss visualization code.

In [ ]:
# Task1: Plot Training Loss Curve
# Using plot_loss_curves from my_utils

print("Task1: Training Loss Curve Visualization")
print("="*50)
print("The plot_loss_curves function has been imported from my_utils.")
print("Usage:")
print("  plot_loss_curves(train_losses, val_accuracies, 'task1_loss_curve.png')")
print("")
print("Function signature:")
print("  plot_loss_curves(train_losses, val_accuracies=None, save_path='loss_curve.png')")
print("")
print("Parameters:")
print("  - train_losses: List of training losses per epoch")
print("  - val_accuracies: List of validation accuracies per epoch (optional)")
print("  - save_path: Path to save the plot image")
print("")
print("plot_loss_curves has already been called in the training cell above.")

Please include the loss curve during training in your report. Does a smaller training loss always mean a better model?

# Task2: Add Regularization

- **L2 Regularization**: Consult PyTorch's [SGD optimizer documentation](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html#sgd) or other online resources. Modify the code in `3. Model Training and Evaluation Process` to add L2 regularization to the loss function, and describe what you changed in your report.
- **Dropout Regularization**: Consult PyTorch's [Dropout layer documentation](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html#dropout) or other online resources. Modify the code in `2. Define the Network Architecture for Classification` to add a Dropout layer **between the first and second linear layers**, and describe what you changed in your report.
- Briefly describe the principles of both regularization methods in your report.

In [ ]:
# Dropout_Net: LeNet with dropout layer added
class Dropout_Net(nn.Module):
    """LeNet variant with dropout regularization.
    
    Adds dropout layer between fc1 and fc2 to prevent overfitting.
    """
    
    def __init__(self, dropout_rate: float = 0.5) -> None:
        # Subclasses of nn.Module must call the parent class constructor
        super(Dropout_Net, self).__init__()

        # Convolutional layer: '3' input channels (RGB), '6' output channels, '5' kernel size
        self.conv1 = nn.Conv2d(3, 6, 5) 
        # Convolutional layer
        self.conv2 = nn.Conv2d(6, 16, 5) 
        # Fully connected layers: y = Wx + b
        self.fc1   = nn.Linear(16*5*5, 120) 
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)
        
        # Dropout layer
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with dropout between fc1 and fc2.
        
        Args:
            x: Input tensor of shape (batch_size, 3, 32, 32).
            
        Returns:
            Output tensor of shape (batch_size, 10).
        """
        # Convolution -> ReLU -> MaxPool
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2) 
        # Flatten; '-1' means automatically inferred size
        x = x.view(x.size()[0], -1) 
        x = F.relu(self.fc1(x))
        # Dropout between fc1 and fc2
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)        
        return x
    
dropout_net = Dropout_Net(dropout_rate=0.5)
print(dropout_net)

In [ ]:
# Task2: Train model with L2 regularization and Dropout

# Set device
device = get_device()

# Create Dropout_Net instance and move to device
dropout_net = Dropout_Net(dropout_rate=0.5)
dropout_net = dropout_net.to(device)

# Define optimizer with L2 regularization (via weight_decay)
# Using Adam optimizer; weight_decay implements L2 regularization
optimizer_dropout = torch.optim.Adam(dropout_net.parameters(), lr=0.001, weight_decay=0.001)

# Create learning rate scheduler (linear warm-up + cosine annealing with restarts)
scheduler_config_dropout = create_learning_rate_scheduler(
    optimizer_dropout, scheduler_type='cosine',
    total_epochs=num_epochs, initial_lr=0.001
)

# Train Dropout network
save_path_dropout = 'checkpoints/lenet_dropout'
print("Training Dropout_Net with L2 regularization...")
train_losses_dropout, val_accuracies_dropout = train(
    trainloader, dropout_net, num_epochs, criterion, optimizer_dropout, 
    save_path_dropout, valloader=valloader, early_stopping_patience=10,
    scheduler_config=scheduler_config_dropout
)

# Plot loss curves
plot_loss_curves(train_losses_dropout, val_accuracies_dropout, 'task2_dropout_loss_curve.png')

# Evaluate on test set
print("\nEvaluating Dropout_Net on test set...")
test_accuracy_dropout = predict(testloader, dropout_net)

# Comparison with baseline model
print("\n=== Task2 Results Comparison ===")
print("Dropout + L2 Regularization Model:")
if val_accuracies_dropout:
    print(f"  - Final validation accuracy: {val_accuracies_dropout[-1]:.2f}%")
else:
    print("  - Final validation accuracy: N/A")
print(f"  - Test accuracy: {test_accuracy_dropout:.2f}%")

# Task3: Hyperparameter Tuning

In the `3. Model Training and Evaluation Process` section, we defined several hyperparameters (such as `num_epochs`, the optimizer's `lr`). Adjust these parameters, observe the changes in the loss function and the model's performance on the test set, briefly describe these changes in your report, and analyze how these hyperparameters affect the model.

In [ ]:
# Task3: Hyperparameter Tuning Experiments
import itertools
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import copy

# Hyperparameter grid (based on DEVELOP_PLAN.md)
hyperparams = {
    'lr': [0.01, 0.001],
    'weight_decay': [0, 1e-4],
    'label_smoothing': [0, 0.05]
}
batch_size = 64  # Fixed, consistent with earlier

print("="*70)
print("Task3: Hyperparameter Tuning Experiments")
print(f"Hyperparameter grid: {hyperparams}")
print(f"Batch size: {batch_size} (fixed)")
print(f"Total experiments: {len(hyperparams['lr']) * len(hyperparams['weight_decay']) * len(hyperparams['label_smoothing'])}")
print("="*70)

# Results storage
results = []

# Device setup
device = get_device()
print(f"Using device: {device}\n")

# Iterate through all hyperparameter combinations
for exp_idx, (lr, wd, ls) in enumerate(itertools.product(
    hyperparams['lr'], hyperparams['weight_decay'], hyperparams['label_smoothing']
), 1):
    print(f"\n{'='*60}")
    print(f"Experiment {exp_idx}/8: lr={lr}, weight_decay={wd}, label_smoothing={ls}")
    print(f"{'='*60}")
    
    # Create new data loader (using same transform)
    trainloader_exp = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    
    # Initialize new model (LeNet)
    model = Net()
    model = model.to(device)
    
    # Configure optimizer (Adam, per development plan)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    
    # Configure loss function (with label smoothing)
    criterion = nn.CrossEntropyLoss(label_smoothing=ls)
    
    # Create learning rate scheduler (linear warm-up + cosine annealing with restarts)
    scheduler_config = create_learning_rate_scheduler(
        optimizer, scheduler_type='cosine',
        total_epochs=num_epochs, initial_lr=lr
    )
    
    # Train model (using my_utils.train_model)
    save_path = f'checkpoints/task3/exp_{exp_idx}'
    train_losses, val_accuracies = train_model(
        model=model,
        train_loader=trainloader_exp,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=num_epochs,
        save_path=save_path,
        val_loader=valloader,
        early_stopping_patience=10,
        gradient_clip=1.0,
        print_every=1000,
        scheduler_config=scheduler_config
    )
    
    # Record results
    final_val_acc = val_accuracies[-1] if val_accuracies else 0
    final_train_loss = train_losses[-1] if train_losses else float('inf')
    best_val_acc = max(val_accuracies) if val_accuracies else 0
    epochs_trained = len(train_losses)
    
    results.append({
        'experiment_id': exp_idx,
        'learning_rate': lr,
        'weight_decay': wd,
        'label_smoothing': ls,
        'batch_size': batch_size,
        'final_val_accuracy': final_val_acc,
        'final_train_loss': final_train_loss,
        'best_val_accuracy': best_val_acc,
        'epochs_trained': epochs_trained,
        'optimizer': 'Adam'
    })
    
    # Save checkpoint
    torch.save(model.state_dict(), f'checkpoints/task3_lr{lr}_wd{wd}_ls{ls}.pth')
    print(f"  Best validation accuracy: {best_val_acc:.2f}%, Final validation accuracy: {final_val_acc:.2f}%")
    print(f"  Epochs trained: {epochs_trained}/{num_epochs}")

# Create results DataFrame
results_df = pd.DataFrame(results)

print("\n" + "="*70)
print("Hyperparameter Tuning Results (sorted by best validation accuracy):")
print("="*70)
print(results_df.sort_values('best_val_accuracy', ascending=False).to_string())

# Save results to CSV
results_df.to_csv('task3_hyperparameter_results.csv', index=False)
print(f"\nResults saved to 'task3_hyperparameter_results.csv'")

# Visualize results
import matplotlib.pyplot as plt

# Plot effects of different hyperparameters
plt.figure(figsize=(15, 5))

# Subplot 1: Effect of learning rate
plt.subplot(1, 3, 1)
for lr in hyperparams['lr']:
    lr_data = results_df[results_df['learning_rate'] == lr]
    plt.bar([str(x) for x in lr_data['experiment_id']], lr_data['best_val_accuracy'], 
            alpha=0.7, label=f'lr={lr}')
plt.xlabel('Experiment ID')
plt.ylabel('Best Validation Accuracy (%)')
plt.title('Effect of Learning Rate')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Effect of weight decay
plt.subplot(1, 3, 2)
for wd in hyperparams['weight_decay']:
    wd_data = results_df[results_df['weight_decay'] == wd]
    if len(wd_data) > 0:
        plt.bar([str(x) for x in wd_data['experiment_id']], wd_data['best_val_accuracy'],
                alpha=0.7, label=f'wd={wd}')
plt.xlabel('Experiment ID')
plt.ylabel('Best Validation Accuracy (%)')
plt.title('Effect of Weight Decay')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 3: Effect of label smoothing
plt.subplot(1, 3, 3)
for ls in hyperparams['label_smoothing']:
    ls_data = results_df[results_df['label_smoothing'] == ls]
    if len(ls_data) > 0:
        plt.bar([str(x) for x in ls_data['experiment_id']], ls_data['best_val_accuracy'],
                alpha=0.7, label=f'ls={ls}')
plt.xlabel('Experiment ID')
plt.ylabel('Best Validation Accuracy (%)')
plt.title('Effect of Label Smoothing')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task3_hyperparameter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Output best hyperparameter combination
best_exp = results_df.loc[results_df['best_val_accuracy'].idxmax()]
print("\n" + "="*70)
print("Best Hyperparameter Combination:")
print("="*70)
print(f"  Experiment ID: {best_exp['experiment_id']}")
print(f"  Learning Rate: {best_exp['learning_rate']}")
print(f"  Weight Decay: {best_exp['weight_decay']}")
print(f"  Label Smoothing: {best_exp['label_smoothing']}")
print(f"  Best Validation Accuracy: {best_exp['best_val_accuracy']:.2f}%")
print(f"  Final Validation Accuracy: {best_exp['final_val_accuracy']:.2f}%")
print(f"  Epochs Trained: {best_exp['epochs_trained']}")
print("="*70)

# Task4: Implement Your Own Network

Consult references (e.g., [Dive into Deep Learning](https://d2l.ai/) and [torchvision model source code](https://github.com/pytorch/vision/tree/main/torchvision/models)). Modify the code in `2. Define the Network Architecture for Classification` to implement a modern convolutional neural network. How does your network compare to the basic LeNet in terms of performance and training time?

In [ ]:
# Task4: Implement Modern CNN (manual ResNet-18)
import torch
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    """Basic residual block for ResNet-18."""
    expansion = 1
    
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, downsample: Optional[nn.Module] = None) -> None:
        super().__init__()
        # First convolution
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        # Second convolution
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
        self.stride = stride
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        if self.downsample is not None:
            identity = self.downsample(x)
            
        out += identity
        out = F.relu(out)
        
        return out

class MyCNN(nn.Module):
    """Manually implemented ResNet-18 for CIFAR-10."""
    
    def __init__(self, num_classes: int = 10, dropout_rate: float = 0.3) -> None:
        super().__init__()
        
        # Initial convolution adapted for 32x32 CIFAR-10 images
        # Original ResNet-18 uses 7x7 stride=2, but we use 3x3 stride=1 for small images
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # ResNet-18 layers: 4 stages with [2, 2, 2, 2] basic blocks
        self.in_channels = 64
        
        # Stage 1: 64 -> 64, no downsampling in first block
        self.layer1 = self._make_layer(64, 2, stride=1)
        # Stage 2: 64 -> 128, downsampling in first block
        self.layer2 = self._make_layer(128, 2, stride=2)
        # Stage 3: 128 -> 256, downsampling in first block
        self.layer3 = self._make_layer(256, 2, stride=2)
        # Stage 4: 256 -> 512, downsampling in first block
        self.layer4 = self._make_layer(512, 2, stride=2)
        
        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Dropout before classifier
        self.dropout = nn.Dropout(p=dropout_rate)
        
        # Final classifier
        self.fc = nn.Linear(512 * BasicBlock.expansion, num_classes)
        
    def _make_layer(self, out_channels: int, blocks: int, stride: int = 1) -> nn.Sequential:
        """Create a layer with specified number of basic blocks."""
        downsample = None
        
        # Create downsample layer if stride != 1 or channels change
        if stride != 1 or self.in_channels != out_channels * BasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * BasicBlock.expansion,
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * BasicBlock.expansion)
            )
            
        layers = []
        # First block in the layer (may have downsampling)
        layers.append(BasicBlock(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * BasicBlock.expansion
        
        # Remaining blocks in the layer (no downsampling)
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_channels, out_channels))
            
        return nn.Sequential(*layers)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Initial convolution
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        # Note: No maxpool here to preserve spatial dimensions for 32x32 input
        
        # Residual stages
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # Global average pooling
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Dropout before classifier
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

# Create MyCNN instance and view parameter count
device = get_device()
mycnn = MyCNN(num_classes=10, dropout_rate=0.3)
mycnn = mycnn.to(device)

param_counts = count_parameters(mycnn)

print("="*70)
print("Task4: Modern Convolutional Neural Network (Manual ResNet-18)")
print("="*70)
print(mycnn)
print(f"\nTotal parameters: {param_counts['total']:,}")
print(f"Trainable parameters: {param_counts['trainable']:,}")
print(f"Using device: {device}")

# Train MyCNN
print("\nStarting MyCNN training...")

# Use best hyperparameters from Task3 (or defaults)
optimizer_mycnn = torch.optim.Adam(mycnn.parameters(), lr=0.001, weight_decay=0.001)
criterion_mycnn = nn.CrossEntropyLoss(label_smoothing=0.05)

# Increase epochs as modern CNNs need more iterations to converge
num_epochs_mycnn = 50

# Create learning rate scheduler (linear warm-up + cosine annealing with restarts, T_0=25 for half of 50 epochs)
scheduler_config_mycnn = create_learning_rate_scheduler(
    optimizer_mycnn, scheduler_type='cosine',
    total_epochs=num_epochs_mycnn, initial_lr=0.001,
    T_0=25, T_mult=2
)

# Train model (using my_utils.train_model)
save_path_mycnn = 'checkpoints/mycnn_resnet18'
train_losses_mycnn, val_accuracies_mycnn = train_model(
    model=mycnn,
    train_loader=trainloader,
    criterion=criterion_mycnn,
    optimizer=optimizer_mycnn,
    num_epochs=num_epochs_mycnn,
    save_path=save_path_mycnn,
    val_loader=valloader,
    early_stopping_patience=15,
    gradient_clip=1.0,
    print_every=1000,
    scheduler_config=scheduler_config_mycnn
)

# Plot loss curves
plot_loss_curves(train_losses_mycnn, val_accuracies_mycnn, 'task4_mycnn_loss_curve.png')

# Evaluate on test set
print("\nEvaluating MyCNN on test set...")
test_accuracy_mycnn = predict(testloader, mycnn)

# Comparison with baseline model
print("\n" + "="*70)
print("Task4 Results: MyCNN (ResNet-18) vs Baseline LeNet")
print("="*70)
print("MyCNN (ResNet-18):")
if val_accuracies_mycnn:
    print(f"  - Best validation accuracy: {max(val_accuracies_mycnn):.2f}%")
    print(f"  - Final validation accuracy: {val_accuracies_mycnn[-1]:.2f}%")
else:
    print("  - Best validation accuracy: N/A")
    print("  - Final validation accuracy: N/A")
print(f"  - Test accuracy: {test_accuracy_mycnn:.2f}%")
print(f"  - Epochs trained: {len(train_losses_mycnn)}")
print(f"  - Total parameters: {param_counts['total']:,}")

print("\nAnalysis:")
print("1. Parameter count: MyCNN (~11M) is far larger than LeNet (~60K)")
print("2. Training time: MyCNN requires more time, but accuracy should be significantly higher")
print("3. Regularization: MyCNN includes BatchNorm and Dropout to help prevent overfitting")
print("4. Expected: MyCNN should achieve 85-90% accuracy on CIFAR-10")
print("="*70)